In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely as shp
from shapely.geometry import Polygon, Point, MultiLineString, LineString
from pyproj import Geod

import osmium as omi
from osmium import FileProcessor as FProc
from pyrosm import OSM

from collections import defaultdict
import warnings
import re

pd.set_option('display.max_columns', None)

g = Geod(ellps="WGS84")
# Function to parse direction,camera:direction tags
def parsedir(strdirection):
    if pd.isna(strdirection):
        return []
    cleaned_string = re.sub(r"[a-zA-Z\.'\" !]", "", str(strdirection)).replace("-", ";").replace(",", ";").replace(":", ";").replace("|", ";")
    directions = [int(d) for d in cleaned_string.split(";") if d and d.isdigit()]
    return directions

# Function to generate trapezoids (ALPR "sightline")
def trapezoidshp(startlng, startlat, direction, height, angle, left_bias, base_width) -> Polygon:
    direction-=left_bias
    hypotenuse = height / np.cos(np.radians((angle/2)))
    bearings = np.array([direction+90, direction-90,direction - angle/2, direction + angle/2])
    lng, lat, _ = g.fwd([startlng]*4, [startlat]*4, bearings, [base_width/2]*2+[hypotenuse]*2)
    return Polygon([(lng[0], lat[0]),(lng[1], lat[1]), (lng[2], lat[2]), (lng[3], lat[3])])

# Function to split "other_tags" into their own columns
def regparse_othertags(row):
    regpat = r'"([^"]+)"=>"([^"]+)"'
    matches = re.findall(regpat, row)
    result = {key: value for key, value in matches}
    return pd.Series(result)

In [ ]:
filefullarea = "./pbf/atlanta261004.osm.pbf"
filesurv = "./pbf/atlanta261004-surveillance.osm.pbf"
fileroad = "./pbf/atlanta261004-roads.osm.pbf"



surveillance = gpd.read_file(filesurv, layer="points")
surveillance = surveillance.fillna("")
surveillance = surveillance.drop("other_tags",axis=1).join(surveillance["other_tags"].apply(regparse_othertags), rsuffix="_fake")
alpr = surveillance[surveillance["surveillance:type"]=="ALPR"].reset_index(drop=True)
alpr = alpr.rename(columns={"direction":"camdir","camera:direction":"camdir2"})
alpr = alpr.iloc[:,:10].merge(alpr.loc[:,["camdir","camdir2"]], left_index=True, right_index=True)
alpr.to_crs("EPSG:4326", inplace=True)
alpr["direction1"] = alpr["camdir"].apply(parsedir)
alpr["direction2"] = alpr["camdir2"].apply(parsedir)

In [ ]:
# Trapezoid projection
angle = 45
visrange = 30
base_width = 10
left_bias = 0

shapelist = []
directionless = gpd.GeoDataFrame()

for n, cam in alpr.iterrows():
    directions = []
    if cam["direction1"]:
        directions = cam["direction1"]
    elif cam["direction2"]:
        directions = cam["direction2"]
    if any(x > 360 for x in directions):
        directions = []

    if directions:
        for direction in directions:
            shape = trapezoidshp(cam.geometry.x, cam.geometry.y, direction, visrange, angle, left_bias, base_width)
            shapelist.append({"camindex":n, "direction":direction, "osmid": cam.osm_id, "geometry": shape})
    else:
        circ = gpd.GeoSeries([Point(cam.geometry.x, cam.geometry.y)])
        buffcirc = gpd.GeoSeries(circ, crs="EPSG:4326").to_crs("EPSG:3857").buffer(visrange, quad_segs=4).to_crs("EPSG:4326")
        shapelist.append({"camindex":n, "osmid":cam.osm_id, "geometry":buffcirc.iloc[0]})

alprtrap = gpd.GeoDataFrame(shapelist, crs="EPSG:4326")
alprtrap = alprtrap.drop_duplicates().reset_index(drop=True)

In [ ]:
roads = gpd.read_file(fileroad, layer="lines")
rmhighway = ["footway","path","cycleway","steps","track","corridor", "pedestrian","bridleway"]
roads = roads[~roads.highway.isin(rmhighway)]
roads = roads.reset_index(drop=True)
alpr2 = alprtrap.copy()
alpr2["geometry"] = alpr2.geometry.exterior
intersects = gpd.overlay(alpr2, roads, how="intersection", keep_geom_type=False).explode()
roads["changed"] = False


In [ ]:
# functions from claude to interpolate/interject intersection points into the geometry. Faster than mvp I wrote, 'tis good enough
def insert_points(line, points, tol=1e-8):
    """Return a new LineString with points inserted at their correct
    arc-length position. Original vertices are untouched."""
    coords = list(line.coords)
    entries = [(line.project(Point(c)), c) for c in coords]

    for p in points:
        d = line.project(p)
        snapped = line.interpolate(d)
        entries.append((d, snapped.coords[0]))

    entries.sort(key=lambda e: e[0])

    result = [entries[0][1]]
    last_d = entries[0][0]
    for d, c in entries[1:]:
        if d - last_d > tol:
            result.append(c)
            last_d = d
    return LineString(result)

def add_intersections_to_roads(roads, points, max_distance=1e-7):
    """
    roads: GeoDataFrame of LineString
    points: GeoDataFrame of Point (already intersected with roads)
    max_distance: matching tolerance in the CRS's units (degrees if
        unprojected lat/lon -- ~1e-5 deg is roughly 1m). Real-world
        intersection geometry is rarely bit-exact on the line, so an
        exact "intersects" join will silently match nothing.
    Returns roads with intersection points inserted as vertices.
    """
    roads = roads.copy()
    joined = gpd.sjoin_nearest(
        points, roads[["geometry"]], max_distance=max_distance, how="inner"
    )
    grouped = joined.groupby("index_right")["geometry"].apply(list)

    roads["geometry"] = roads.apply(
        lambda row: insert_points(row.geometry, grouped.get(row.name, [])),
        axis=1,
    )
    return roads

rr = add_intersections_to_roads(roads, intersects)
rr.loc[rr.osm_id.isin(intersects.osm_id),"changed"] = True


In [ ]:
outputfile = "./bbr-weights.osm.pbf"

from collections import defaultdict
gatetags = {"barrier":"gate"}
gate2tags = {"barrier":"lift_gate"}
# rr["geometry"] = rr.geometry.line_merge()
multiids = set(rr[rr.geometry.geom_type=="MultiLineString"].osm_id.astype(int))
editids = set(rr[rr.changed].osm_id.astype(int)) - multiids

fakeid = (2**44)
bothfilter = omi.filter.IdFilter(editids|multiids)
roadsfilter = omi.filter.KeyFilter("highway")

nodegates = defaultdict(int)
firstpass = FProc(filefullarea, omi.osm.WAY|omi.osm.NODE)\
        .with_locations()\
        .with_filter(roadsfilter)\
        .with_filter(bothfilter)
for obj in firstpass:
    if obj.is_way():
        if obj.id in multiids:
            pass
        elif obj.id in editids:
            newwaynodes = []
            row = rr.loc[rr.osm_id==str(obj.id)].iloc[0]
            if len([x for x in obj.nodes])>2:
                refs = [nod.ref for nod in obj.nodes][1:-1]
                clon = [nod.lon for nod in obj.nodes][1:-1]
                clat = [nod.lat for nod in obj.nodes][1:-1]
                newcoords = [(lon, lat) for lon, lat in zip(*row.geometry.xy)][1:-1]
            else:
                refs = [nod.ref for nod in obj.nodes]
                clon = [nod.lon for nod in obj.nodes]
                clat = [nod.lat for nod in obj.nodes]
                newcoords = [(lon, lat) for lon, lat in zip(*row.geometry.xy)]

            lref = refs[0]
            for newlon, newlat in newcoords:
                _, _, distances = g.inv([newlon]*len(clon), [newlat]*len(clat), clon, clat)
                closest = min(distances)
                if closest >= 0.01:
                    lref = refs[distances.index(closest)]
                    newwaynodes.append(lref)
                    nodegates[lref] += 1

ng = dict(nodegates)
nodeids = set(ng)
nodefilter = omi.filter.IdFilter(nodeids)

with omi.SimpleWriter(outputfile, overwrite=True) as writer:
    nodes = FProc(filefullarea, omi.osm.NODE|omi.osm.WAY|omi.osm.RELATION)\
            .with_locations()\
            .with_filter(nodefilter)\
            .handler_for_filtered(writer)
    for obj in nodes:
        if obj.is_node():
            if ng.get(obj.id) > 1:
                ntags = dict(obj.tags)
                ntags.update(gatetags)
                writer.add_node(omi.osm.mutable.Node(id=obj.id, 
                    location=(obj.lon, obj.lat), tags=ntags))
            else:
                ntags = dict(obj.tags)
                ntags.update(gate2tags)
                writer.add_node(omi.osm.mutable.Node(id=obj.id, 
                    location=(obj.lon, obj.lat), tags=ntags))
    writer.close()